Training with cross-validation for the Approximate Nearest Neighbors (ANN) with Hassanat distance metric base model. ANN was used instead of k-nearest neighbors (KNN) because KNN is too computationally expensive for the available resources. Hassanat distance was used instead of Euclidean distance because it captures rarer target classes more effectively. The ANN model predicts the target class (Drug) from the features (age, race/ethnicity, sex, family income, insurance coverage, prescription strength, prescription day supply, prescription quantity, and prescription form).

In [ ]:
# Install required libraries.
# scikit-learn: KNN model, preprocessing, CV, metrics.
# permetrics: macro/micro F2 score and other evaluation metrics.
!pip install scikit-learn permetrics pynndescent

In [ ]:
# Import all required libraries.
import pandas as pd
import numpy as np
import sklearn
import joblib
from sklearn.preprocessing import LabelEncoder, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from pynndescent import PyNNDescentTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report
)
from sklearn.utils import shuffle
from permetrics import ClassificationMetric
from numba import njit
import warnings
warnings.filterwarnings('ignore')

# Confirm versions for reproducibility
print('scikit-learn version:', sklearn.__version__)
print('pandas version:', pd.__version__)
print('numpy version:', np.__version__)

# OneHotEncoder parameter name changed from 'sparse' to 'sparse_output' in sklearn 1.2.
# This flag ensures the notebook runs on both older and newer sklearn versions.
from packaging import version
OHE_SPARSE_KWARG = 'sparse_output' if version.parse(sklearn.__version__) >= version.parse('1.2') else 'sparse'

# Numba-optimized Hassanat Distance
@njit
def hassanat_distance(a, b):
    total = 0.0
    for i in range(a.shape[0]):
        x = a[i]
        y = b[i]

        min_val = min(x, y)
        max_val = max(x, y)

        if min_val >= 0:
            total += 1.0 - (1.0 + min_val) / (1.0 + max_val)
        else:
            shift = abs(min_val)
            total += 1.0 - 1.0 / (1.0 + max_val + shift)

    return total

In [ ]:
# Load the super integrated dataset (2014-2021).
# The super dataset contains demographics + prescription features + Person_ID.

DATA_DIR = './'  # change this to your data path if needed

super_df = pd.read_csv(
    f'{DATA_DIR}super_integrated_data.csv',
    sep=None,
    engine='python',
    encoding='utf-8-sig'
)

if 'Unnamed: 0' in super_df.columns:
    super_df = super_df.drop(columns=['Unnamed: 0'])

print('Super dataset shape:', super_df.shape)
print('\nSuper dataset columns:', super_df.columns.tolist())
print('\nMissing values:')
print(super_df.isnull().sum())

In [ ]:
# Verify Person_ID is present in the dataset.
# Person_ID is used only for StratifiedGroupKFold grouping — not a model feature.
assert 'Person_ID' in super_df.columns, \
    "ERROR: Person_ID not found in dataset — check super_integrated_data.csv."
assert super_df['Person_ID'].isnull().sum() == 0, \
    "ERROR: Missing Person_IDs — check super_integrated_data.csv."

print('Person_ID verified.')
print('Unique persons:', super_df['Person_ID'].nunique())

In [ ]:
# Prescription NaN handling — fill with -1 sentinel.
# The 97,497 no-prescription rows have NaN for Quantity/Strength/Day_Supply/Form
# because those fields are structurally absent (not missing at random).
# Filling with -1 preserves this as a meaningful signal distinct from real values.
# After RobustScaler, -1 maps to a distinct region in feature space.
#
# Age, Strength, and Day_Supply real missing values are handled inside the CV loop
# using hierarchical median imputation on the train split only — see the CV cell below.

numeric_prescription_cols = ['Quantity', 'Strength', 'Day_Supply']
mask = super_df['Drug'] == 'no prescriptions'

# Form is a categorical string column (values like TABS, ORAL, CAPS).
# Fill with string '-1' so it is treated as a distinct 'no prescription' category
# by the OneHotEncoder later.
super_df.loc[mask, 'Form'] = super_df.loc[mask, 'Form'].fillna('-1')

print('Missing values after prescription fill:')
print(super_df[['Quantity', 'Form', 'Strength', 'Day_Supply']].isnull().sum())
print('\nRemaining demographic NaNs (will be imputed inside CV loop):')
print(super_df[['Age', 'Family_income']].isnull().sum())

In [ ]:
# Imputation functions for Age, Strength, and Day_Supply.
# Applied within each fold to prevent data leakage.
# Consistent with all other base models.

def fit_age_medians(df):
    df = df.copy()

    income_bin_edges = {}
    income_bracket = pd.Series(index=df.index, dtype='float64')
    for year, group in df.groupby('Year'):
        try:
            bins, edges = pd.qcut(
                group['Family_income'], 4, labels=False,
                duplicates='drop', retbins=True
            )
            income_bin_edges[year] = edges
            income_bracket.loc[group.index] = bins + 1
        except ValueError:
            income_bin_edges[year] = None
    df['income_bracket'] = income_bracket

    hh_medians    = df.groupby(['Year', 'Household_ID'])['Age'].median()
    grp_medians   = df.groupby(['Year', 'income_bracket', 'Insurance_coverage'])['Age'].median()
    yr_medians    = df.groupby('Year')['Age'].median()
    global_median = df['Age'].median()

    return {
        'income_bin_edges': income_bin_edges,
        'hh_medians': hh_medians,
        'grp_medians': grp_medians,
        'yr_medians': yr_medians,
        'global_median': global_median,
        'available_years': sorted(yr_medians.index.unique().tolist()),
    }

def _nearest_year(year, available_years):
    if year in available_years:
        return year
    return min(available_years, key=lambda y: abs(y - year))

def _assign_income_bracket(df, income_bin_edges, available_years):
    income_bracket = pd.Series(index=df.index, dtype='float64')
    for year, group in df.groupby('Year'):
        effective_year = _nearest_year(year, available_years)
        edges = income_bin_edges.get(effective_year)
        if edges is not None:
            binned = pd.cut(group['Family_income'], bins=edges, labels=False, include_lowest=True)
            income_bracket.loc[group.index] = binned + 1
    return income_bracket

def apply_age_medians(df, medians):
    df = df.copy()
    available_years = medians['available_years']

    df['income_bracket'] = _assign_income_bracket(df, medians['income_bin_edges'], available_years)

    # Map every row's Year to its nearest training year for all lookups below.
    effective_year = df['Year'].apply(lambda y: _nearest_year(y, available_years))

    hh_idx    = pd.MultiIndex.from_arrays([effective_year, df['Household_ID']])
    hh_lookup = pd.Series(hh_idx.map(medians['hh_medians']), index=df.index)

    grp_idx    = pd.MultiIndex.from_arrays([effective_year, df['income_bracket'], df['Insurance_coverage']])
    grp_lookup = pd.Series(grp_idx.map(medians['grp_medians']), index=df.index)

    yr_lookup = effective_year.map(medians['yr_medians'])

    df['Age'] = df['Age'].fillna(hh_lookup)
    df['Age'] = df['Age'].fillna(grp_lookup)
    df['Age'] = df['Age'].fillna(yr_lookup)
    df['Age'] = df['Age'].fillna(medians['global_median'])
    df.drop(columns=['income_bracket'], inplace=True)
    return df

def fit_strength_medians(df):
    mask = df['Drug'] != 'no prescriptions'
    yr_drug_medians = df.loc[mask].groupby(['Year', 'Drug'])['Strength'].median()
    drug_medians    = df.loc[mask].groupby('Drug')['Strength'].median()
    global_median   = df.loc[mask, 'Strength'].median()
    return {
        'yr_drug_medians': yr_drug_medians,
        'drug_medians': drug_medians,
        'global_median': global_median,
    }

def apply_strength_medians(df, medians):
    df = df.copy()
    mask = df['Drug'] != 'no prescriptions'

    yr_drug_idx = pd.MultiIndex.from_frame(df.loc[mask, ['Year', 'Drug']])
    yr_drug_lookup = pd.Series(yr_drug_idx.map(medians['yr_drug_medians']), index=df.loc[mask].index)
    drug_lookup = df.loc[mask, 'Drug'].map(medians['drug_medians'])

    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(yr_drug_lookup)
    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(drug_lookup)
    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(medians['global_median'])
    return df

def fit_day_supply_medians(df):
    mask = df['Drug'] != 'no prescriptions'
    yr_drug_medians = df.loc[mask].groupby(['Year', 'Drug'])['Day_Supply'].median()
    drug_medians    = df.loc[mask].groupby('Drug')['Day_Supply'].median()
    global_median   = df.loc[mask, 'Day_Supply'].median()
    return {
        'yr_drug_medians': yr_drug_medians,
        'drug_medians': drug_medians,
        'global_median': global_median,
    }

def apply_day_supply_medians(df, medians):
    df = df.copy()
    mask = df['Drug'] != 'no prescriptions'

    yr_drug_idx = pd.MultiIndex.from_frame(df.loc[mask, ['Year', 'Drug']])
    yr_drug_lookup = pd.Series(yr_drug_idx.map(medians['yr_drug_medians']), index=df.loc[mask].index)
    drug_lookup = df.loc[mask, 'Drug'].map(medians['drug_medians'])

    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(yr_drug_lookup)
    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(drug_lookup)
    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(medians['global_median'])
    return df

In [ ]:
# Exploratory check before modeling.
# Verify class count, drug distribution, and person count.
print('Unique persons:', super_df['Person_ID'].nunique())
print('Unique drugs:', super_df['Drug'].nunique())
print('No prescription rows:', (super_df['Drug'] == 'no prescriptions').sum())
print('Actual drug rows:', (super_df['Drug'] != 'no prescriptions').sum())

print('\nTop 10 most prescribed drugs:')
print(super_df['Drug'].value_counts().head(10))

print('\nBottom 5 rarest drugs:')
print(super_df['Drug'].value_counts().tail(5))

# Check class imbalance ratio — max count / min count.
# High ratio confirms severe imbalance and justifies Hassanat distance
# and distance-weighted voting in KNN.
counts = super_df['Drug'].value_counts()
print(f'\nMost common drug count:  {counts.max():,}')
print(f'Rarest drug count:       {counts.min():,}')
print(f'Imbalance ratio:         {counts.max() / counts.min():.1f}x')

# Hard stop if drug count is not 217 (216 drugs + "no prescriptions").
assert super_df['Drug'].nunique() == 217, \
    f"ERROR: Expected 217 drug classes, found {super_df['Drug'].nunique()}."

print('\nDrug class count confirmed: 217 (216 drugs + no prescriptions).')

In [ ]:
# Define feature columns, categorical columns, numeric columns, and target.
#
# Categorical features are one-hot encoded (KNN cannot natively handle unordered
# categorical values — ordinal codes would imply false ordinal relationships).
#
# Numeric features are scaled with RobustScaler inside the CV loop
# to prevent scale leakage across folds. RobustScaler is used over StandardScaler
# because it is less sensitive to outliers due to our severe class imbalance.
#
# LabelEncoder is fit once on the full dataset before the CV loop.
# Fitting inside the loop would produce different integer mappings per fold,
# making fold results incomparable and breaking per-drug recall aggregation.

feature_cols     = ['Age', 'Sex', 'Family_income', 'Insurance_coverage',
                    'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']
categorical_cols = ['Sex', 'Insurance_coverage', 'Race_ethnicity', 'Form']
numeric_cols     = ['Age', 'Family_income', 'Quantity', 'Strength', 'Day_Supply']
target_col       = 'Drug'

le = LabelEncoder()
super_df['Drug_encoded'] = le.fit_transform(super_df[target_col])

print('Unique classes:', len(le.classes_))
print('Feature cols:', feature_cols)
print('Categorical cols:', categorical_cols)
print('Numeric cols:', numeric_cols)
print('\nSample drug to integer mapping (first 5):')
for i, drug in enumerate(le.classes_[:5]):
    print(f'  {drug} -> {i}')

# Hard stop if any defined column is missing from the dataset.
# Catches typos in column names or dataset changes before the CV loop starts.
missing_cols = [c for c in feature_cols + [target_col] if c not in super_df.columns]
assert len(missing_cols) == 0, \
    f"ERROR: These columns are missing from the dataset: {missing_cols}"

# Save the LabelEncoder so it can be reloaded without rerunning this notebook.
# Required for ensemble model — all models must use identical drug-to-integer mappings.
joblib.dump(le, 'knn_super_label_encoder.joblib')
print('\nLabelEncoder saved to knn_super_label_encoder.joblib')

In [ ]:
# define class to scale numerics without -1 sentinel value interference

class SelectiveRobustScaler(BaseEstimator, TransformerMixin):
    def __init__(self, drug_encoded_val, sentinel_value=-1.0):
        self.drug_encoded_val = drug_encoded_val
        self.sentinel_value = sentinel_value
        self.scaler = RobustScaler()

    def fit(self, X, y=None):
        # We only fit the scaler on rows that are NOT the sentinel class
        # Pipeline passes 'y' to the fit method of transformers.
        if y is not None:
            mask = (y != self.drug_encoded_val)
            if mask.any():
                self.scaler.fit(X[mask])
            else:
                self.scaler.fit(X) # Fallback if no rows match
        return self

    def transform(self, X):
        # RobustScaler handles NaNs by leaving them as NaNs.
        # We transform everything, then fill the NaNs with our sentinel.
        X_scaled = self.scaler.transform(X)
        X_scaled = np.nan_to_num(X_scaled, nan=self.sentinel_value)
        return X_scaled

In [ ]:
# Define the preprocessing pipeline.
#
# Numeric pipeline:
#   RobustScaler — scales using median and IQR instead of mean and std.
#   More robust to outliers than StandardScaler, which is important given
#   our severe class imbalance where rare drug classes can produce outlier values.
#   Applied before Hassanat distance computation to ensure all features
#   contribute equally when distances are aggregated across dimensions.
#   Fitted on train data only inside each fold to prevent scale leakage.
#   Note: Age, Strength, and Day_Supply are imputed before this pipeline runs
#   using hierarchical median imputation functions — SimpleImputer is not used.
#
# Categorical pipeline:
#   OneHotEncoder — converts categories to binary indicator columns.
#   handle_unknown='ignore' safely handles rare categories in val
#   not seen during training.
#   sparse_output=False returns a dense array (required for KNN with
#   a custom distance metric like Hassanat).
#
# NOTE: The preprocessor is fit on train data only inside each fold.
# It is NOT fit here — fitting here would leak val statistics into
# scaler medians/IQRs, inflating CV performance.

# Get the encoded value for "no prescriptions"
no_presc_val = list(le.classes_).index('no prescriptions')

# Define the three branches
preprocessor = ColumnTransformer(transformers=[
    # 1. Demographic Numeric (Standard Scaling)
    ('demog_num', RobustScaler(), ['Age', 'Family_income']),

    # 2. Prescription Numeric (Target-Aware Scaling + Sentinel)
    ('presc_num', SelectiveRobustScaler(drug_encoded_val=no_presc_val),
     ['Quantity', 'Strength', 'Day_Supply']),

    # 3. Categorical (One-Hot Encoding)
    ('cat', OneHotEncoder(**{OHE_SPARSE_KWARG: False}, handle_unknown='ignore'),
     categorical_cols)
])

print('Preprocessor defined.')
print('  Numeric pipeline:     RobustScaler')
print('  Categorical pipeline: OneHotEncoder')
print('  Note: preprocessor is not fitted here — fitted inside each CV fold on train split only.')

In [ ]:
# 5-fold stratified group cross-validation.
#
# StratifiedGroupKFold ensures:
#   1. Each fold has similar drug class distribution (stratified)
#   2. All records for the same person stay in the same fold (grouped)
#      preventing the model from seeing the same person in both train and val
#
# For each fold:
#   1. Split by Person_ID groups and drug label stratification
#   2. Impute Age, Strength, Day_Supply within each fold (fit on train, apply to val)
#   3. Fit RobustScaler and OneHotEncoder on train split only — apply to both train and val
#   4. Train KNN with Hassanat distance on preprocessed train fold
#   5. Predict on preprocessed val fold
#   6. Compute and store all metrics + per-drug recall

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

X      = super_df[feature_cols].copy()
y      = super_df['Drug_encoded'].values
groups = super_df['Person_ID'].values

fold_results          = []
per_drug_recall_folds = []

for fold_num, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups), start=1):
    print(f'\n{"="*50}')
    print(f'FOLD {fold_num}/5')
    print(f'{"="*50}')

    train_fold = super_df.iloc[train_idx].copy()
    val_fold   = super_df.iloc[val_idx].copy()

    # Impute Age, Strength, Day_Supply within each fold.
    # Imputation is applied on train and val separately to prevent data leakage.
    age_medians = fit_age_medians(train_fold)
    train_fold = apply_age_medians(train_fold, age_medians)
    val_fold   = apply_age_medians(val_fold, age_medians)

    strength_medians = fit_strength_medians(train_fold)
    train_fold = apply_strength_medians(train_fold, strength_medians)
    val_fold   = apply_strength_medians(val_fold, strength_medians)

    day_supply_medians = fit_day_supply_medians(train_fold)
    train_fold = apply_day_supply_medians(train_fold, day_supply_medians)
    val_fold   = apply_day_supply_medians(val_fold, day_supply_medians)

    X_train_fold = train_fold[feature_cols].copy()
    X_val_fold   = val_fold[feature_cols].copy()
    y_train_fold = train_fold['Drug_encoded'].values
    y_val_fold   = val_fold['Drug_encoded'].values

    print(f'Train size: {len(X_train_fold):,} | Val size: {len(X_val_fold):,}')
    print(f'Unique drugs in train: {len(np.unique(y_train_fold))} | val: {len(np.unique(y_val_fold))}')

    # Fit preprocessor on train split only.
    # RobustScaler and OneHotEncoder statistics are computed from train data only —
    # no leakage from val into preprocessing.
    X_train_processed = preprocessor.fit_transform(X_train_fold, y_train_fold)
    X_val_processed   = preprocessor.transform(X_val_fold)

    print(f'Preprocessed train shape: {X_train_processed.shape}')

    # The ANN Indexer (Builds the graph using Hassanat)
    ann_index = PyNNDescentTransformer(
        n_neighbors=5,
        metric=hassanat_distance,
        n_jobs=-1,
        random_state=42,
        low_memory=False,
        parallel_batch_queries=True
    )

    # The Classifier (Uses the precomputed distances from the indexer)
    knn_clf = KNeighborsClassifier(
        n_neighbors=5,
        weights='distance',
        metric='precomputed'
    )

    # Combine into a Pipeline
    model = Pipeline([
        ('ann', ann_index),
        ('clf', knn_clf)
    ])

    model.fit(X_train_processed, y_train_fold)
    print(f'Fold {fold_num} training complete.')

    y_pred_fold = model.predict(X_val_processed)

    # Overall metrics.
    acc   = accuracy_score(y_val_fold, y_pred_fold)
    kappa = cohen_kappa_score(y_val_fold, y_pred_fold)
    mcc   = matthews_corrcoef(y_val_fold, y_pred_fold)

    # Macro and micro averaged metrics via permetrics.
    # Macro: average per class equally — treats rare and common drugs equally.
    # Micro: aggregate all counts — dominated by the most common drugs.
    evaluator = ClassificationMetric(y_val_fold, y_pred_fold)

    macro_precision = evaluator.precision_score(average='macro')
    micro_precision = evaluator.precision_score(average='micro')
    macro_recall    = evaluator.recall_score(average='macro')
    micro_recall    = evaluator.recall_score(average='micro')
    macro_f1        = evaluator.f1_score(average='macro')
    micro_f1        = evaluator.f1_score(average='micro')

    # F2 score weights recall twice as much as precision.
    # Higher recall is more important here because missing a drug class
    # means underestimating its wastewater load.
    macro_f2 = evaluator.fbeta_score(beta=2, average='macro')
    micro_f2 = evaluator.fbeta_score(beta=2, average='micro')

    fold_results.append({
        'fold': fold_num,
        'accuracy': acc,
        'cohen_kappa': kappa,
        'mcc': mcc,
        'macro_precision': macro_precision,
        'micro_precision': micro_precision,
        'macro_recall': macro_recall,
        'micro_recall': micro_recall,
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'macro_f2': macro_f2,
        'micro_f2': micro_f2,
    })

    # Per-drug recall for this fold.
    # Used for ensemble model selection — the ensemble picks the best model per drug.
    report = classification_report(
        y_val_fold, y_pred_fold,
        labels=np.arange(len(le.classes_)),
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )
    drug_recalls         = {drug: report[drug]['recall'] for drug in le.classes_ if drug in report}
    drug_recalls['fold'] = fold_num
    per_drug_recall_folds.append(drug_recalls)

    print(f'Fold {fold_num} results:')
    print(f'  Accuracy:      {acc:.4f}')
    print(f'  Cohen Kappa:   {kappa:.4f}')
    print(f'  MCC:           {mcc:.4f}')
    print(f'  Macro Recall:  {macro_recall:.4f}')
    print(f'  Micro Recall:  {micro_recall:.4f}')
    print(f'  Macro F2:      {macro_f2:.4f}')

print('\n' + '='*50)
print('ALL FOLDS COMPLETE')
print('='*50)

In [ ]:
# Summarize cross-validation results across all 5 folds.
# Report mean and standard deviation for each metric.
# Standard deviation shows how stable the model is across different data splits.
results_df = pd.DataFrame(fold_results)

print('CV RESULTS — MEAN +/- STD ACROSS 5 FOLDS')
print('='*55)
metric_cols = [c for c in results_df.columns if c != 'fold']
for col in metric_cols:
    mean = results_df[col].mean()
    std  = results_df[col].std()
    print(f'  {col:<25}: {mean:.4f} +/- {std:.4f}')

results_df.to_csv('knn_super_cv_results.csv', index=False)
print('\nCV results saved to knn_super_cv_results.csv')

# Quick sanity check — all 5 folds must be present before summarizing.
# If the CV loop crashed mid-run, this catches it before saving incomplete results.
assert len(results_df) == 5, \
    f"ERROR: Expected 5 fold results, found {len(results_df)} — CV may not have completed."

print('All 5 folds confirmed complete.')

In [ ]:
# Compute average per-drug recall across all 5 folds.
# This is the key output for ensemble model selection.
# The ensemble picks whichever base model has the highest recall for each drug.
per_drug_df = pd.DataFrame(per_drug_recall_folds)
drug_cols   = [c for c in per_drug_df.columns if c != 'fold']

mean_drug_recall = per_drug_df[drug_cols].mean().reset_index()
mean_drug_recall.columns = ['Drug', 'Mean_Recall_KNN_Super']
mean_drug_recall = mean_drug_recall.sort_values('Mean_Recall_KNN_Super', ascending=False)

print('Top 20 drugs by mean recall:')
print(mean_drug_recall.head(20).to_string(index=False))

print('\nBottom 20 drugs by mean recall:')
print(mean_drug_recall.tail(20).to_string(index=False))

# Summary stats — how many drugs does KNN recall at all?
# A drug is considered recalled if mean recall > 0 across folds.
# This number goes directly into the paper for model comparison.
drugs_recalled = (mean_drug_recall['Mean_Recall_KNN_Super'] > 0).sum()
print(f'\nDrugs with mean recall > 0: {drugs_recalled} / {len(mean_drug_recall)}')
print(f'Drugs with mean recall >= 0.1: {(mean_drug_recall["Mean_Recall_KNN_Super"] >= 0.1).sum()}')
print(f'Drugs with mean recall >= 0.5: {(mean_drug_recall["Mean_Recall_KNN_Super"] >= 0.5).sum()}')

mean_drug_recall.to_csv('knn_super_per_drug_recall.csv', index=False)
print('\nPer-drug recall saved to knn_super_per_drug_recall.csv')

In [ ]:
# Train final model on the full training dataset (2014-2021).
# After CV gives confidence in model performance, retrain on all available data
# to maximize signal before evaluating on the held-out 2022 validation set.
#
# The preprocessor is fit on the full training data here (no fold split).
# The fitted preprocessor is reused to transform 2022 data in the next cell.

# Impute Age, Strength, Day_Supply on full dataset before final training.
age_medians_final = fit_age_medians(super_df)
strength_medians_final = fit_strength_medians(super_df)
day_supply_medians_final = fit_day_supply_medians(super_df)

joblib.dump(age_medians_final, 'knn_super_age_medians.joblib')
joblib.dump(strength_medians_final, 'knn_super_strength_medians.joblib')
joblib.dump(day_supply_medians_final, 'knn_super_day_supply_medians.joblib')

data_final = apply_age_medians(super_df.copy(), age_medians_final)
data_final = apply_strength_medians(data_final, strength_medians_final)
data_final = apply_day_supply_medians(data_final, day_supply_medians_final)

X_final = data_final[feature_cols].copy()
y_final = data_final['Drug_encoded'].values

# Shuffle to separate refill records after imputation.
X_final, y_final = shuffle(X_final, y_final, random_state=42)
X_final = X_final.reset_index(drop=True)

print('Final training data size:', X_final.shape)
print('Number of classes:', len(le.classes_))
print('\nFitting preprocessor on full training data...')

X_final_processed = preprocessor.fit_transform(X_final, y_final)
print('Preprocessed shape:', X_final_processed.shape)

print('\nTraining final KNN model with Hassanat distance...')

# The ANN Indexer (Builds the graph using Hassanat)
ann_index = PyNNDescentTransformer(
    n_neighbors=5,
    metric=hassanat_distance,
    n_jobs=-1,
    random_state=42,
    low_memory=False,
    parallel_batch_queries=True
)

# The Classifier (Uses the precomputed distances from the indexer)
knn_clf = KNeighborsClassifier(
    n_neighbors=5,
    weights='distance',
    metric='precomputed'
)

# Combine into a Pipeline
final_model = Pipeline([
    ('ann', ann_index),
    ('clf', knn_clf)
])

final_model.fit(X_final_processed, y_final)
print('Final model training complete.')

# Extract the fitted transformer step from the pipeline
fitted_ann_step = final_model.named_steps['ann']

# Extract the index object itself from inside the transformer step
# This extracts the underlying compiled search graph structure
pynndescent_index = fitted_ann_step.index_

# Save final model and preprocessor so the notebook does not need to be
# rerun if the kernel dies. Preprocessor must be saved alongside the model
# because it must be applied identically to 2022 data in the next cell.
joblib.dump(preprocessor, 'knn_super_preprocessor.joblib')
joblib.dump(final_model.named_steps['clf'], 'knn_super_clf_only.joblib')
joblib.dump(pynndescent_index, 'knn_final_prebuilt_index.joblib')
print('joblib files saved successfully!')